In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import root_mean_squared_log_error
import os
from sklearn.model_selection import KFold

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/sample_submission.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/metadata.csv
/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv


In [2]:
# ==========================================================
# CELL 1 : IMPORT LIBRARIES & CONFIGURATION
# ==========================================================
import warnings
warnings.filterwarnings('ignore')

import os
import random
import time

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer  # Added for SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

from catboost import CatBoostRegressor
import lightgbm as lgb
import xgboost as xgb

# ----------------------------------------------------------
# RANDOM SEED
# ----------------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# ----------------------------------------------------------
# COMPETITION PATHS
# ----------------------------------------------------------

TRAIN_PATH = "/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/train.csv"
TEST_PATH = "/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/test.csv"
SAMPLE_PATH = "/kaggle/input/competitions/heavy-equipment-selling-price-prediction-challenge/sample_submission.csv"
OUTPUT_PATH = "/kaggle/working/submission.csv"

TARGET = "TargetValue"
ID_COL = "TransactionID"
N_FOLDS = 5

# ----------------------------------------------------------
# K-FOLD
# ----------------------------------------------------------

kf = KFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

print("=" * 70)
print("Heavy Equipment Selling Price Prediction")
print("=" * 70)

print(f"Random Seed : {SEED}")
print(f"Number of Folds : {N_FOLDS}")
print()
print("Libraries Imported Successfully")









    


    

Heavy Equipment Selling Price Prediction
Random Seed : 42
Number of Folds : 5

Libraries Imported Successfully


In [3]:
# ==========================================================
# CELL 2 : LOAD DATA & BASIC PREPROCESSING
# ==========================================================

print("=" * 70)
print("LOADING DATA")
print("=" * 70)

# ----------------------------------------------------------
# Load Dataset
# ----------------------------------------------------------

train = pd.read_csv(TRAIN_PATH, low_memory=False)
test = pd.read_csv(TEST_PATH, low_memory=False)
sample = pd.read_csv(SAMPLE_PATH)

print("Train Shape :", train.shape)
print("Test Shape  :", test.shape)

# Save Test IDs
test_ids = test[ID_COL].copy()

# ----------------------------------------------------------
# Target
# ----------------------------------------------------------

y = np.log1p(train[TARGET])
train = train.drop(columns=[TARGET])

# ----------------------------------------------------------
# Merge Train & Test
# ----------------------------------------------------------

train["is_train"] = 1
test["is_train"] = 0

full = pd.concat(
    [train, test],
    axis=0,
    ignore_index=True
)

print("Combined Shape :", full.shape)

# ----------------------------------------------------------
# Remove Sparse Columns (>99.5% Missing)
# ----------------------------------------------------------

missing_ratio = full.isnull().mean()
drop_cols = missing_ratio[missing_ratio > 0.995].index.tolist()

if len(drop_cols) > 0:
    print("\nDropping Sparse Columns")
    print(drop_cols)
    full.drop(columns=drop_cols, inplace=True)

# ----------------------------------------------------------
# Remove Constant Columns
# ----------------------------------------------------------

constant_cols = []
for col in full.columns:
    if full[col].nunique(dropna=False) <= 1:
        constant_cols.append(col)

if len(constant_cols) > 0:
    print("\nDropping Constant Columns")
    print(constant_cols)
    full.drop(columns=constant_cols, inplace=True)

# ----------------------------------------------------------
# Identify Feature Types
# ----------------------------------------------------------

cat_cols = full.select_dtypes(include="object").columns.tolist()
num_cols = full.select_dtypes(exclude="object").columns.tolist()

if ID_COL in num_cols:
    num_cols.remove(ID_COL)

# ----------------------------------------------------------
# Handle Missing Values via SimpleImputer
# ----------------------------------------------------------

# Impute categorical features with a constant value
if len(cat_cols) > 0:
    cat_imputer = SimpleImputer(strategy="constant", fill_value="Missing")
    full[cat_cols] = cat_imputer.fit_transform(full[cat_cols])

# Impute numerical features with their median value
if len(num_cols) > 0:
    num_imputer = SimpleImputer(strategy="median")
    full[num_cols] = num_imputer.fit_transform(full[num_cols])

print()
print("Categorical Features :", len(cat_cols))
print("Numeric Features     :", len(num_cols))
print()
print("Missing Values Remaining :", full.isnull().sum().sum())
print()
print("Current Shape :", full.shape)

LOADING DATA
Train Shape : (138701, 50)
Test Shape  : (15000, 49)
Combined Shape : (153701, 50)

Dropping Sparse Columns
['col18', 'col19']

Categorical Features : 42
Numeric Features     : 5

Missing Values Remaining : 0

Current Shape : (153701, 48)


In [4]:
#==========================================================
# CELL 3 : ADVANCED FEATURE ENGINEERING
# ==========================================================

print("=" * 70)
print("ADVANCED FEATURE ENGINEERING")
print("=" * 70)

# ----------------------------------------------------------
# Transaction Date Features
# ----------------------------------------------------------

full["TransactionDate"] = pd.to_datetime(full["TransactionDate"])

full["SaleYear"] = full["TransactionDate"].dt.year
full["SaleMonth"] = full["TransactionDate"].dt.month
full["SaleDay"] = full["TransactionDate"].dt.day
full["SaleWeek"] = full["TransactionDate"].dt.isocalendar().week.astype(int)
full["SaleQuarter"] = full["TransactionDate"].dt.quarter
full["SaleDayOfWeek"] = full["TransactionDate"].dt.dayofweek
full["SaleDayOfYear"] = full["TransactionDate"].dt.dayofyear

full["IsWeekend"] = (full["SaleDayOfWeek"] >= 5).astype(int)
full["IsMonthStart"] = (full["SaleDay"] <= 3).astype(int)
full["IsMonthEnd"] = (full["SaleDay"] >= 28).astype(int)

full.drop(columns=["TransactionDate"], inplace=True)

# ----------------------------------------------------------
# Manufacture Year
# ----------------------------------------------------------

valid_year = full.loc[full["ManufactureYear"] >= 1900, "ManufactureYear"].median()
full["MissingYear"] = (full["ManufactureYear"] < 1900).astype(int)
full.loc[full["ManufactureYear"] < 1900, "ManufactureYear"] = valid_year

# ----------------------------------------------------------
# Machine Age
# ----------------------------------------------------------

full["MachineAge"] = (full["SaleYear"] - full["ManufactureYear"]).clip(lower=0)

# ----------------------------------------------------------
# Hours Features
# ----------------------------------------------------------

full["OperationalHoursMeter"] = full["OperationalHoursMeter"].fillna(0)
full["LogHours"] = np.log1p(full["OperationalHoursMeter"])
full["HoursPerYear"] = full["OperationalHoursMeter"] / (full["MachineAge"] + 1)
full["HoursPerYear"] = full["HoursPerYear"].replace(np.inf, 0).fillna(0)

# ----------------------------------------------------------
# Interaction Features
# ----------------------------------------------------------

full["Age_x_Hours"] = full["MachineAge"] * full["OperationalHoursMeter"]
full["Age_div_Hours"] = full["MachineAge"] / (full["OperationalHoursMeter"] + 1)
full["WearIndex"] = full["MachineAge"] + full["OperationalHoursMeter"] / 1000

# ----------------------------------------------------------
# Polynomial Features
# ----------------------------------------------------------

full["AgeSquared"] = full["MachineAge"] ** 2
full["HoursSquared"] = full["OperationalHoursMeter"] ** 2
full["LogAge"] = np.log1p(full["MachineAge"])

# ----------------------------------------------------------
# Cyclic Month Features
# ----------------------------------------------------------

full["MonthSin"] = np.sin(2 * np.pi * full["SaleMonth"] / 12)
full["MonthCos"] = np.cos(2 * np.pi * full["SaleMonth"] / 12)

# ----------------------------------------------------------
# Frequency Encoding
# ----------------------------------------------------------

freq_cols = [
    "ProductConfigID",
    "AssetID",
    "VendorPartnerID",
    "Spec_BaseClass",
    "FunctionalClassification",
    "RegionCode"
]

for col in freq_cols:
    if col in full.columns:
        freq = full[col].value_counts()
        full[col + "_Freq"] = full[col].map(freq)

# ----------------------------------------------------------
# Count Encoding
# ----------------------------------------------------------

count_cols = [
    "InventoryGroupCategory",
    "InventoryGroupDescription",
    "CabinType"
]

for col in count_cols:
    if col in full.columns:
        full[col + "_Count"] = full.groupby(col)[col].transform("count")

# ----------------------------------------------------------
# High Cardinality Frequency Encoding
# ----------------------------------------------------------

cat_cols = full.select_dtypes(include="object").columns.tolist()

for col in cat_cols:
    if full[col].nunique() > 30:
        freq = full[col].value_counts()
        full[col + "_FE"] = full[col].map(freq)

print()
print("Feature Engineering Completed")
print("Current Shape :", full.shape)

ADVANCED FEATURE ENGINEERING

Feature Engineering Completed
Current Shape : (153701, 86)


In [5]:
# ==========================================================
# CELL 4 : LABEL ENCODING & TRAIN-TEST SPLIT
# ==========================================================

print("=" * 70)
print("LABEL ENCODING")
print("=" * 70)

# ----------------------------------------------------------
# Identify Categorical Columns
# ----------------------------------------------------------

cat_cols = full.select_dtypes(include="object").columns.tolist()
print("Categorical Columns :", len(cat_cols))

# ----------------------------------------------------------
# Label Encoding
# ----------------------------------------------------------

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    full[col] = le.fit_transform(full[col].astype(str))
    encoders[col] = le

print("Encoding Completed")

# ----------------------------------------------------------
# Split Train & Test
# ----------------------------------------------------------

train = full[full["is_train"] == 1].copy()
test = full[full["is_train"] == 0].copy()

train.drop(columns=["is_train"], inplace=True)
test.drop(columns=["is_train"], inplace=True)

X = train.drop(columns=[ID_COL])
X_test = test.drop(columns=[ID_COL])

print()
print("Train Shape :", X.shape)
print("Test Shape  :", X_test.shape)
print()
print("Feature Count :", X.shape[1])

LABEL ENCODING
Categorical Columns : 41
Encoding Completed

Train Shape : (138701, 84)
Test Shape  : (15000, 84)

Feature Count : 84


In [6]:
# ==========================================================
# CELL 5 : CATBOOST MODEL
# ==========================================================

print("=" * 70)
print("TRAINING CATBOOST")
print("=" * 70)

oof_cb = np.zeros(len(X))
test_cb = np.zeros((len(X_test), N_FOLDS))
scores_cb = []

cb_params = {

    "loss_function": "RMSE",

    "eval_metric": "RMSE",

    "iterations": 12000,

    "learning_rate": 0.015,

    "depth": 8,

    "l2_leaf_reg": 10,

    "random_strength": 1,

    "bootstrap_type": "Bernoulli",

    "subsample": 0.80,

    "grow_policy": "SymmetricTree",

    "task_type": "GPU",

    "devices": "0",

    "early_stopping_rounds": 500,

    "random_seed": SEED,

    "verbose": False

}

for fold, (train_idx, valid_idx) in enumerate(kf.split(X), 1):
    print("=" * 60)
    print(f"CATBOOST FOLD {fold}")
    print("=" * 60)

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostRegressor(**cb_params)
    model.fit(
        X_train,
        y_train,
        eval_set=(X_valid, y_valid),
        use_best_model=True,
        verbose=False
    )

    valid_pred = model.predict(X_valid)
    test_pred = model.predict(X_test)

    oof_cb[valid_idx] = valid_pred
    test_cb[:, fold - 1] = test_pred

    rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))
    scores_cb.append(rmse)
    print(f"Fold RMSE : {rmse:.6f}")

catboost_test_prediction = test_cb.mean(axis=1)

print()
print("Average CatBoost RMSE :", np.mean(scores_cb))
print("CatBoost Training Completed")


# ==========================================================

TRAINING CATBOOST
CATBOOST FOLD 1
Fold RMSE : 0.204643
CATBOOST FOLD 2
Fold RMSE : 0.207397
CATBOOST FOLD 3
Fold RMSE : 0.207465
CATBOOST FOLD 4
Fold RMSE : 0.208650
CATBOOST FOLD 5
Fold RMSE : 0.205622

Average CatBoost RMSE : 0.20675551799529698
CatBoost Training Completed


In [7]:
# ==========================================================
# CELL 6 : LIGHTGBM MODEL
# ==========================================================

print("=" * 70)
print("TRAINING LIGHTGBM")
print("=" * 70)

oof_lgb = np.zeros(len(X))
test_lgb = np.zeros((len(X_test), N_FOLDS))
scores_lgb = []

lgb_params = {

    "objective": "regression",

    "metric": "rmse",

    "learning_rate": 0.02,

    "num_leaves": 255,

    "max_depth": -1,

    "feature_fraction": 0.80,

    "bagging_fraction": 0.80,

    "bagging_freq": 5,

    "min_child_samples": 20,

    "lambda_l1": 1,

    "lambda_l2": 8,

    "device": "gpu",

    "seed": SEED,

    "verbosity": -1

}

for fold, (train_idx, valid_idx) in enumerate(kf.split(X), 1):
    print("=" * 60)
    print(f"LIGHTGBM FOLD {fold}")
    print("=" * 60)

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    train_set = lgb.Dataset(X_train, label=y_train)
    valid_set = lgb.Dataset(X_valid, label=y_valid)

    model = lgb.train(
        params=lgb_params,
        train_set=train_set,
        num_boost_round=12000,
        valid_sets=[valid_set],
        callbacks=[
            lgb.early_stopping(500),
            lgb.log_evaluation(0)
        ]
    )

    valid_pred = model.predict(X_valid, num_iteration=model.best_iteration)
    test_pred = model.predict(X_test, num_iteration=model.best_iteration)

    oof_lgb[valid_idx] = valid_pred
    test_lgb[:, fold - 1] = test_pred

    rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))
    scores_lgb.append(rmse)
    print(f"Fold RMSE : {rmse:.6f}")

lgb_test_prediction = test_lgb.mean(axis=1)

print()
print("Average LightGBM RMSE :", np.mean(scores_lgb))
print("LightGBM Training Completed")

TRAINING LIGHTGBM
LIGHTGBM FOLD 1


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


Training until validation scores don't improve for 500 rounds
Early stopping, best iteration is:
[6333]	valid_0's rmse: 0.199883
Fold RMSE : 0.199883
LIGHTGBM FOLD 2
Training until validation scores don't improve for 500 rounds
Early stopping, best iteration is:
[5365]	valid_0's rmse: 0.202215
Fold RMSE : 0.202215
LIGHTGBM FOLD 3
Training until validation scores don't improve for 500 rounds
Early stopping, best iteration is:
[5610]	valid_0's rmse: 0.201842
Fold RMSE : 0.201842
LIGHTGBM FOLD 4
Training until validation scores don't improve for 500 rounds
Early stopping, best iteration is:
[5612]	valid_0's rmse: 0.20404
Fold RMSE : 0.204040
LIGHTGBM FOLD 5
Training until validation scores don't improve for 500 rounds
Early stopping, best iteration is:
[5203]	valid_0's rmse: 0.20086
Fold RMSE : 0.200860

Average LightGBM RMSE : 0.20176797905569605
LightGBM Training Completed


In [8]:
# ==========================================================
# CELL 7 : XGBOOST MODEL
# ==========================================================

print("=" * 70)
print("TRAINING XGBOOST")
print("=" * 70)

oof_xgb = np.zeros(len(X))
test_xgb = np.zeros((len(X_test), N_FOLDS))
scores_xgb = []

xgb_params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "learning_rate": 0.02,
    "max_depth": 8,
    "min_child_weight": 5,
    "subsample": 0.85,
    "colsample_bytree": 0.80,
    "colsample_bylevel": 0.80,
    "gamma": 0.10,
    "lambda": 10.0,
    "alpha": 2.0,
    "max_bin": 256,
    "tree_method": "hist",
    "seed": SEED
}

for fold, (train_idx, valid_idx) in enumerate(kf.split(X), 1):
    print("=" * 60)
    print(f"XGBOOST FOLD {fold}")
    print("=" * 60)

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    dtrain = xgb.DMatrix(X_train, label=y_train)
    dvalid = xgb.DMatrix(X_valid, label=y_valid)
    dtest = xgb.DMatrix(X_test)

    model = xgb.train(
        params=xgb_params,
        dtrain=dtrain,
        num_boost_round=12000,
        evals=[(dvalid, "Validation")],
        early_stopping_rounds=500,
        verbose_eval=False
    )

    valid_pred = model.predict(dvalid, iteration_range=(0, model.best_iteration + 1))
    test_pred = model.predict(dtest, iteration_range=(0, model.best_iteration + 1))

    oof_xgb[valid_idx] = valid_pred
    test_xgb[:, fold - 1] = test_pred

    rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))
    scores_xgb.append(rmse)
    print(f"Fold RMSE : {rmse:.6f}")

xgb_test_prediction = test_xgb.mean(axis=1)

print()
print("Average XGBoost RMSE :", np.mean(scores_xgb))
print("XGBoost Training Completed")

TRAINING XGBOOST
XGBOOST FOLD 1
Fold RMSE : 0.208181
XGBOOST FOLD 2
Fold RMSE : 0.210850
XGBOOST FOLD 3
Fold RMSE : 0.210818
XGBOOST FOLD 4
Fold RMSE : 0.212541
XGBOOST FOLD 5
Fold RMSE : 0.208406

Average XGBoost RMSE : 0.2101592167976011
XGBoost Training Completed


In [9]:
# ==========================================================
# CELL 8 : RIDGE STACKING (CV)
# ==========================================================

print("=" * 70)
print("RIDGE STACKING")
print("=" * 70)

# ----------------------------------------------------------
# Create Meta Features
# ----------------------------------------------------------

meta_train = pd.DataFrame({
    "catboost": oof_cb,
    "lightgbm": oof_lgb,
    "xgboost": oof_xgb
})

meta_test = pd.DataFrame({
    "catboost": catboost_test_prediction,
    "lightgbm": lgb_test_prediction,
    "xgboost": xgb_test_prediction
})

print("Meta Train Shape :", meta_train.shape)
print("Meta Test Shape  :", meta_test.shape)

# ----------------------------------------------------------
# Ridge Stacking
# ----------------------------------------------------------

meta_oof_ridge = np.zeros(len(meta_train))
meta_test_ridge = np.zeros((len(meta_test), N_FOLDS))
ridge_scores = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(meta_train), 1):
    print("=" * 60)
    print(f"RIDGE FOLD {fold}")
    print("=" * 60)

    X_train = meta_train.iloc[train_idx]
    X_valid = meta_train.iloc[valid_idx]
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    ridge = Ridge(alpha=1.0)
    ridge.fit(X_train, y_train)

    valid_pred = ridge.predict(X_valid)
    test_pred = ridge.predict(meta_test)

    meta_oof_ridge[valid_idx] = valid_pred
    meta_test_ridge[:, fold - 1] = test_pred

    rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))
    ridge_scores.append(rmse)
    print(f"Fold RMSE : {rmse:.6f}")

ridge_test_prediction = meta_test_ridge.mean(axis=1)

print()
print("Average Ridge RMSE :", np.mean(ridge_scores))
print("Ridge Stacking Completed")

RIDGE STACKING
Meta Train Shape : (138701, 3)
Meta Test Shape  : (15000, 3)
RIDGE FOLD 1
Fold RMSE : 0.197886
RIDGE FOLD 2
Fold RMSE : 0.200708
RIDGE FOLD 3
Fold RMSE : 0.200364
RIDGE FOLD 4
Fold RMSE : 0.202189
RIDGE FOLD 5
Fold RMSE : 0.199195

Average Ridge RMSE : 0.200068511488323
Ridge Stacking Completed


In [10]:
# ==========================================================
# CELL 9 : CATBOOST META MODEL (CV)
# ==========================================================

print("=" * 70)
print("CATBOOST META STACKING")
print("=" * 70)

meta_oof_cb = np.zeros(len(meta_train))
meta_test_cb = np.zeros((len(meta_test), N_FOLDS))
meta_scores_cb = []

meta_params = {
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "iterations": 3000,
    "learning_rate": 0.03,
    "depth": 3,
    "l2_leaf_reg": 8,
    "random_seed": SEED,
    "verbose": False
}

for fold, (train_idx, valid_idx) in enumerate(kf.split(meta_train), 1):
    print("=" * 60)
    print(f"META CATBOOST FOLD {fold}")
    print("=" * 60)

    X_train = meta_train.iloc[train_idx]
    X_valid = meta_train.iloc[valid_idx]
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostRegressor(**meta_params)
    model.fit(
        X_train,
        y_train,
        eval_set=(X_valid, y_valid),
        use_best_model=True,
        verbose=False
    )

    valid_pred = model.predict(X_valid)
    test_pred = model.predict(meta_test)

    meta_oof_cb[valid_idx] = valid_pred
    meta_test_cb[:, fold - 1] = test_pred

    rmse = np.sqrt(mean_squared_error(y_valid, valid_pred))
    meta_scores_cb.append(rmse)
    print(f"Fold RMSE : {rmse:.6f}")

cat_meta_prediction = meta_test_cb.mean(axis=1)

print()
print("Average Meta CatBoost RMSE :", np.mean(meta_scores_cb))
print("CatBoost Meta Model Completed")

CATBOOST META STACKING
META CATBOOST FOLD 1
Fold RMSE : 0.197890
META CATBOOST FOLD 2
Fold RMSE : 0.200542
META CATBOOST FOLD 3
Fold RMSE : 0.200567
META CATBOOST FOLD 4
Fold RMSE : 0.202177
META CATBOOST FOLD 5
Fold RMSE : 0.199252

Average Meta CatBoost RMSE : 0.2000854362039072
CatBoost Meta Model Completed


In [11]:
# ==========================================================
# CELL 10 : OPTIMIZED BLENDING
# ==========================================================

print("=" * 70)
print("OPTIMIZING ENSEMBLE")
print("=" * 70)

# ----------------------------------------------------------
# Search Best Base Blend
# ----------------------------------------------------------

best_rmse = 999
best_weights = None

for cb_w in np.arange(0.10, 0.71, 0.05):
    for lgb_w in np.arange(0.10, 0.71, 0.05):
        xgb_w = 1.0 - cb_w - lgb_w
        if xgb_w < 0:
            continue

        blend_oof = (
            cb_w * oof_cb +
            lgb_w * oof_lgb +
            xgb_w * oof_xgb
        )

        rmse = np.sqrt(mean_squared_error(y, blend_oof))

        if rmse < best_rmse:
            best_rmse = rmse
            best_weights = (cb_w, lgb_w, xgb_w)

print()
print("Best Base Blend RMSE :", best_rmse)
print("Best Base Weights :", best_weights)

# ----------------------------------------------------------
# Test Blend
# ----------------------------------------------------------

blend_test_prediction = (
    best_weights[0] * catboost_test_prediction +
    best_weights[1] * lgb_test_prediction +
    best_weights[2] * xgb_test_prediction
)

blend_oof = (
    best_weights[0] * oof_cb +
    best_weights[1] * oof_lgb +
    best_weights[2] * oof_xgb
)

# ----------------------------------------------------------
# Search Final Ensemble
# ----------------------------------------------------------

best_final_rmse = 999
best_final_weights = None

for ridge_w in np.arange(0.20, 0.61, 0.05):
    for cat_w in np.arange(0.20, 0.61, 0.05):
        blend_w = 1.0 - ridge_w - cat_w
        if blend_w < 0:
            continue

        final_oof = (
            ridge_w * meta_oof_ridge +
            cat_w * meta_oof_cb +
            blend_w * blend_oof
        )

        rmse = np.sqrt(mean_squared_error(y, final_oof))

        if rmse < best_final_rmse:
            best_final_rmse = rmse
            best_final_weights = (ridge_w, cat_w, blend_w)

print()
print("Best Final RMSE :", best_final_rmse)
print("Best Final Weights :", best_final_weights)

# ----------------------------------------------------------
# Final Test Prediction
# ----------------------------------------------------------

final_prediction = (
    best_final_weights[0] * ridge_test_prediction +
    best_final_weights[1] * cat_meta_prediction +
    best_final_weights[2] * blend_test_prediction
)

print()
print("Final Ensemble Ready")

OPTIMIZING ENSEMBLE

Best Base Blend RMSE : 0.20047477558832252
Best Base Weights : (np.float64(0.30000000000000004), np.float64(0.6500000000000001), np.float64(0.04999999999999982))

Best Final RMSE : 0.19988470805287128
Best Final Weights : (np.float64(0.49999999999999994), np.float64(0.49999999999999994), np.float64(5.551115123125783e-17))

Final Ensemble Ready


In [12]:
# ==========================================================
# CELL 11 : CREATE FINAL SUBMISSION
# ==========================================================

print("=" * 70)
print("CREATING FINAL SUBMISSION")
print("=" * 70)

# ----------------------------------------------------------
# Reverse Log Transformation
# ----------------------------------------------------------

final_prediction = np.expm1(final_prediction)

# ----------------------------------------------------------
# Prevent Negative Predictions
# ----------------------------------------------------------

final_prediction = np.clip(final_prediction, 0, None)

# ----------------------------------------------------------
# Create Submission DataFrame
# ----------------------------------------------------------

submission = pd.DataFrame({
    ID_COL: test_ids,
    TARGET: final_prediction
})

# ----------------------------------------------------------
# Save Submission
# ----------------------------------------------------------

submission.to_csv(OUTPUT_PATH, index=False)

print()
print("Submission File Created Successfully!")
print()
print(submission.head())
print()

print("=" * 70)
print("Prediction Statistics")
print("=" * 70)

print(submission[TARGET].describe())
print()
print("Minimum Prediction :", submission[TARGET].min())
print("Maximum Prediction :", submission[TARGET].max())
print("Mean Prediction    :", submission[TARGET].mean())
print()
print("Submission Shape :", submission.shape)
print()
print("Saved At :", OUTPUT_PATH)
print()
print("=" * 70)
print("MODEL TRAINING COMPLETED SUCCESSFULLY")
print("=" * 70)

CREATING FINAL SUBMISSION

Submission File Created Successfully!

   TransactionID   TargetValue
0        1139307  60961.705513
1        1139419  79559.652379
2        1139482  30496.441485
3        1139522  20182.114702
4        1139684  12734.819265

Prediction Statistics
count     15000.000000
mean      40903.564687
std       24958.859932
min        7688.874469
25%       20930.522119
50%       34706.771445
75%       56339.203741
max      137810.012626
Name: TargetValue, dtype: float64

Minimum Prediction : 7688.874469430324
Maximum Prediction : 137810.0126258569
Mean Prediction    : 40903.564687460756

Submission Shape : (15000, 2)

Saved At : /kaggle/working/submission.csv

MODEL TRAINING COMPLETED SUCCESSFULLY
